# Classificação de Dígitos — SVHN

Grupo: Samuel, Wilker e Bernardo.

**Como rodar no Colab:** `Ambiente de execução → Alterar o tipo de ambiente → GPU`, depois `Executar tudo`.

**Protocolo**
- `split="train"` é dividido em 90% treino e 10% validação, com semente fixa (`SEED = 42`).
- A validação acompanha os experimentos: cada experimento é avaliado na época com melhor acurácia de validação.
- `split="test"` só informa a acurácia; não participa de nenhuma escolha.
- Cada experimento muda **uma coisa** em relação a um anterior, para isolar o efeito.
- Vários experimentos e avaliações vêm do conteúdo das aulas (ativações, pré-processamento, loss SVM,
  L2, SGD, kNN, random subsampling e matriz de confusão); a aula de origem aparece em cada seção.

In [ ]:
import io
import json
import math
import os
import shutil
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from PIL import Image
from torch.func import functional_call
from torchvision import datasets

SEED = 42
EPOCHS = 15
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
WEIGHTS_PATH = Path("weights/best.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Opcional: salvar os resultados no Google Drive para não perder nada se o Colab desconectar.
# Experimentos já salvos em RESULTS_DIR são carregados em vez de retreinados.
# from google.colab import drive; drive.mount("/content/drive")
# RESULTS_DIR = Path("/content/drive/MyDrive/svhn/results")
# WEIGHTS_PATH = Path("/content/drive/MyDrive/svhn/weights/best.pt")

# Modo de teste do CI (SVHN_SMOKE=1): dados aleatórios, poucas épocas e saídas em smoke/, sem baixar o SVHN.
# Sem a variável, nada muda.
SMOKE = os.environ.get("SVHN_SMOKE") == "1"
if SMOKE:
    EPOCHS = 2
    RESULTS_DIR, WEIGHTS_PATH = Path("smoke/results"), Path("smoke/weights/best.pt")
    shutil.rmtree("smoke", ignore_errors=True)  # sem cache: o teste sempre treina tudo de novo

# Mesma semente -> mesmos resultados também na GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"device: {DEVICE}" + (" | modo de teste (dados aleatórios)" if SMOKE else ""))

## Dados

O dataset inteiro cabe na memória da GPU (~300 MB em `uint8`), então os batches são montados direto
na GPU, sem `DataLoader`. A média e o desvio da normalização são calculados **só no conjunto de treino**.

In [ ]:
def load_split(split):
    if SMOKE:  # mesmo formato do SVHN: uint8 N x 3 x 32 x 32 e rótulos de 0 a 9
        g = torch.Generator().manual_seed(SEED + ["train", "test"].index(split))
        n = 500 if split == "train" else 200  # pequeno: o teste confere que tudo roda, não a acurácia
        return (torch.randint(0, 256, (n, 3, 32, 32), dtype=torch.uint8, generator=g),
                torch.randint(0, 10, (n,), generator=g))
    try:
        ds = datasets.SVHN(DATA_DIR, split=split, download=True)
    except OSError:  # servidor de Stanford fora do ar: usa o espelho oficial do SVHN no Hugging Face
        return load_split_hf(split)
    return torch.from_numpy(ds.data), torch.from_numpy(ds.labels).long()  # data: N x 3 x 32 x 32, uint8


HF_URL = "https://huggingface.co/datasets/ufldl-stanford/svhn/resolve/main/cropped_digits/{}-00000-of-00001.parquet"


def load_split_hf(split):
    """Mesmo conteúdo e formato do torchvision, lido do parquet (PNG + rótulo de 0 a 9)."""
    path = DATA_DIR / f"{split}.parquet"
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(".tmp")
        urllib.request.urlretrieve(HF_URL.format(split), tmp)
        tmp.replace(path)  # só vira .parquet depois de baixado por inteiro
    df = pd.read_parquet(path)
    x = np.stack([np.asarray(Image.open(io.BytesIO(img["bytes"]))) for img in df["image"]])  # N x 32 x 32 x 3
    return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous(), torch.tensor(df["label"].to_numpy())


x_train_full, y_train_full = load_split("train")
x_test, y_test = (t.to(DEVICE) for t in load_split("test"))


def make_split(seed):
    """Divide o split="train" em 90% treino e 10% validação, sorteado com `seed`. Devolve tensores na GPU."""
    perm = torch.randperm(len(x_train_full), generator=torch.Generator().manual_seed(seed))
    val, tr = perm[: len(perm) // 10], perm[len(perm) // 10:]
    return tuple(t.to(DEVICE) for t in (x_train_full[tr], y_train_full[tr], x_train_full[val], y_train_full[val]))


x_train, y_train, x_val, y_val = make_split(SEED)
GRAY = torch.tensor([0.299, 0.587, 0.114], device=DEVICE).view(1, 3, 1, 1)  # luminância (ITU-R 601)
pixels = x_train.float().div(255)  # ~800 MB: convertido uma vez para as duas estatísticas
STD, MEAN = (t.view(1, 3, 1, 1) for t in torch.std_mean(pixels, dim=(0, 2, 3)))
GRAY_STD, GRAY_MEAN = torch.std_mean((pixels * GRAY).sum(1))
del pixels
DEFAULT_PRE = {"norm": "mean_std", "gray": False}  # pré-processamento padrão (também usado pelo kNN)
print(f"treino {len(x_train)} | validação {len(x_val)} | teste {len(x_test)}")


def preprocess(x, cfg):
    """uint8 -> float. cfg["norm"]: "mean_std" (média e desvio por canal), "mean" (só centraliza, como a
    aula-2 recomenda) ou "none" (só /255). cfg["gray"]: converte para tons de cinza (1 canal) antes de
    normalizar, com a média e o desvio do próprio cinza."""
    x = x.float() / 255
    mean, std = MEAN, STD
    if cfg["gray"]:
        x, mean, std = (x * GRAY).sum(1, keepdim=True), GRAY_MEAN, GRAY_STD
    if cfg["norm"] in ("mean", "mean_std"):
        x = x - mean
    if cfg["norm"] == "mean_std":
        x = x / std
    return x


def augment(x):
    """Rotação de ±8°, escala de ±10% e translação de ±8% aleatórias por imagem, feitas na GPU."""
    n, dev = len(x), x.device
    rand = lambda: torch.rand(n, device=dev) * 2 - 1
    angle, scale = rand() * math.radians(8), 1 + rand() * 0.1
    tx, ty = rand() * 0.16, rand() * 0.16  # o grid vai de -1 a 1, então 8% da largura = 0.16
    cos, sin = torch.cos(angle) / scale, torch.sin(angle) / scale
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, padding_mode="zeros", align_corners=False)  # 0 = cor média após normalizar


def cutout(x, size=8):
    """Zera (= cor média) um quadrado size x size em posição aleatória de cada imagem; pode sair pela borda."""
    n, _, h, w = x.shape
    cy, cx = (torch.randint(0, d, (n, 1, 1), device=x.device) for d in (h, w))
    ys, xs = torch.arange(h, device=x.device).view(1, h, 1), torch.arange(w, device=x.device).view(1, 1, w)
    hole = (ys >= cy - size // 2) & (ys < cy + size // 2) & (xs >= cx - size // 2) & (xs < cx + size // 2)
    return x * ~hole[:, None]

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    axes[0, i].imshow(x_train[i].permute(1, 2, 0).cpu())
    axes[0, i].set_title(int(y_train[i]))
    aug = augment(preprocess(x_train[i:i + 1], DEFAULT_PRE))
    axes[1, i].imshow((aug * STD + MEAN)[0].clamp(0, 1).permute(1, 2, 0).cpu())
for ax in axes.flat:
    ax.axis("off")
axes[1, 0].set_title("augment", loc="left")
plt.show()

## Modelos

In [ ]:
def linear():
    return nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10))


# Ativações da aula-2; a Leaky ReLU segue o slide: max(0.1x, x).
ACTIVATIONS = {"relu": nn.ReLU, "leaky_relu": lambda: nn.LeakyReLU(0.1), "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


# Inicializações comparadas nos extras (#14); None = padrão do PyTorch (Kaiming uniforme com a=√5).
INITS = {"xavier": nn.init.xavier_uniform_,
         "kaiming_normal": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"),
         "normal_0.01": lambda w: nn.init.normal_(w, 0, 0.01)}


def mlp(hidden=(512, 256), dropout=0.0, batchnorm=False, activation="relu", in_features=3 * 32 * 32, init=None):
    layers, size = [nn.Flatten()], in_features
    for h in hidden:
        layers.append(nn.Linear(size, h))
        if batchnorm:
            layers.append(nn.BatchNorm1d(h))
        layers += [ACTIVATIONS[activation](), nn.Dropout(dropout)]
        size = h
    model = nn.Sequential(*layers, nn.Linear(size, 10))
    if init:
        for m in model:
            if isinstance(m, nn.Linear):
                INITS[init](m.weight)
                nn.init.zeros_(m.bias)
    return model


def cnn(widths=(32, 64, 128), dropout=0.3, batchnorm=True, convs=2):
    """3 blocos [conv-BN-ReLU x convs + maxpool]: 32x32 -> 4x4, seguidos de uma camada densa."""
    layers, cin = [], 3
    norm = nn.BatchNorm2d if batchnorm else nn.Identity  # sem BN: extra cnn_sem_bn (#14)
    for w in widths:
        for _ in range(convs):
            layers += [nn.Conv2d(cin, w, 3, padding=1), norm(w), nn.ReLU()]
            cin = w
        layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers, nn.Flatten(), nn.Dropout(dropout), nn.Linear(cin * 4 * 4, 256), nn.ReLU(),
                         nn.Dropout(dropout), nn.Linear(256, 10))

## Treino

AdamW (com `weight_decay=0` equivale ao Adam) ou SGD com mini-batch (aula de backpropagation), e LR com
decaimento cosseno ao longo das épocas. A loss de treino pode ser a softmax (entropia cruzada) ou a SVM
multiclasse (aula de loss), com regularização L2 opcional somada à loss: L = perda + λ·Σ W².
A loss de validação é sempre a entropia cruzada pura, para comparar todos os experimentos.

In [ ]:
def batches(x, y, size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    n = len(x) - 1 if shuffle and len(x) % size == 1 else len(x)  # preventivo: lote de 1 quebra o BatchNorm1d das MLPs
    for i in range(0, n, size):
        j = idx[i:i + size]
        yield x[j], y[j]


def svm_loss(scores, y):
    """Loss SVM multiclasse da aula: L_i = soma, para j != y_i, de max(0, s_j - s_y_i + 1); média no batch."""
    margins = (scores - scores.gather(1, y[:, None]) + 1).clamp(min=0)
    return (margins * (1 - F.one_hot(y, scores.shape[1]))).sum(1).mean()


def run_epoch(model, x, y, cfg, opt=None):
    """Uma passada sobre (x, y). Treina se `opt` for dado; senão só avalia. Retorna (loss, acurácia)."""
    training = opt is not None
    model.train(training)
    total_loss, correct = torch.zeros((), device=x.device), torch.zeros((), device=x.device)
    with torch.set_grad_enabled(training):
        for xb, yb in batches(x, y, cfg["batch"] if training else 1024, shuffle=training):
            xb = preprocess(xb, cfg)
            if training and cfg["augment"]:
                xb = augment(xb)
            if training and cfg.get("cutout"):  # .get: chave nova fora dos padrões, não invalida o cache
                xb = cutout(xb)
            out = model(xb)
            if not training:
                loss = F.cross_entropy(out, yb)
            elif cfg["loss"] == "svm":
                loss = svm_loss(out, yb)
            else:
                loss = F.cross_entropy(out, yb, label_smoothing=cfg["label_smoothing"])
            if training and cfg["l2"]:
                loss = loss + cfg["l2"] * sum(p.pow(2).sum() for p in model.parameters() if p.dim() > 1)
            if training:
                opt.zero_grad()
                loss.backward()
                opt.step()
            total_loss += loss.detach() * len(yb)
            correct += (out.argmax(1) == yb).sum()
    return total_loss.item() / len(y), correct.item() / len(y)


ARCHS = {"linear": linear, "mlp": mlp, "cnn": cnn}


def full_config(cfg):
    """Completa `cfg` com os valores padrão. A config completa é a chave do cache de resultados."""
    return {"arch_args": {}, "batch": 64, "epochs": EPOCHS, "augment": False, "weight_decay": 0.0,
            "label_smoothing": 0.0, "seed": SEED, **DEFAULT_PRE, "loss": "ce",
            "l2": 0.0, "optimizer": "adamw", "momentum": 0.0, **cfg}


def setup(cfg):
    """Modelo, otimizador e LR cosseno ao longo de cfg["epochs"], com a semente da config."""
    torch.manual_seed(cfg["seed"])
    model = ARCHS[cfg["arch"]](**cfg["arch_args"]).to(DEVICE)
    if cfg["optimizer"] == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=cfg["momentum"],
                              weight_decay=cfg["weight_decay"])
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    return model, opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"])


def train(cfg, split=None):
    """Treina conforme `cfg` (já completa) e devolve (modelo na melhor época de validação, histórico, resumo).

    `split` = (x_treino, y_treino, x_val, y_val); por padrão, a divisão fixa da semente SEED.
    """
    x_tr, y_tr, x_v, y_v = split or (x_train, y_train, x_val, y_val)
    if "train_frac" in cfg:  # curva de aprendizado (#14): x_tr já está embaralhado, o começo é uma amostra
        n = round(len(x_tr) * cfg["train_frac"])
        x_tr, y_tr = x_tr[:n], y_tr[:n]
    epochs = cfg["epochs"]
    model, opt, sched = setup(cfg)
    hist = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}
    best_val, best_epoch, best_state = -1.0, 0, None
    start = time.time()
    for ep in range(1, epochs + 1):
        run_epoch(model, x_tr, y_tr, cfg, opt)
        sched.step()
        # Métricas de treino medidas em modo de avaliação (sem dropout/augment), comparáveis às de validação.
        tl, ta = run_epoch(model, x_tr, y_tr, cfg)
        vl, va = run_epoch(model, x_v, y_v, cfg)
        for k, v in zip(hist, (tl, ta, vl, va)):
            hist[k].append(v)
        if va > best_val:
            best_val, best_epoch = va, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"  ep {ep:2d}/{epochs} | treino loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}")
    model.load_state_dict(best_state)
    _, test_acc = run_epoch(model, x_test, y_test, cfg)
    b = best_epoch - 1
    summary = dict(epocas=epochs, melhor_epoca=best_epoch,
                   train_loss=hist["train_loss"][b], val_loss=hist["val_loss"][b],
                   train_acc=hist["train_acc"][b], val_acc=best_val, test_acc=test_acc,
                   tempo_s=round(time.time() - start))
    if cfg["loss"] == "svm":  # a SVM também reporta a própria loss (hinge) na validação
        with torch.no_grad():
            model.eval()
            summary["val_loss_svm"] = svm_loss(model(preprocess(x_v, cfg)), y_v).item()
    return model, hist, summary


def train_final(cfg):
    """Treina em todo o split="train" (treino + validação) até cfg["stop_epoch"].

    O LR segue o mesmo cosseno de cfg["epochs"] épocas do experimento, interrompido na época da melhor
    validação, para repetir o caminho de otimização que a validação escolheu. Fica o modelo da última época.
    """
    x_tr, y_tr = torch.cat([x_train, x_val]), torch.cat([y_train, y_val])
    model, opt, sched = setup(cfg)
    start = time.time()
    for ep in range(1, cfg["stop_epoch"] + 1):
        run_epoch(model, x_tr, y_tr, cfg, opt)
        sched.step()
        print(f"  ep {ep:2d}/{cfg['stop_epoch']}")
    train_loss, train_acc = run_epoch(model, x_tr, y_tr, cfg)
    _, test_acc = run_epoch(model, x_test, y_test, cfg)
    return model, dict(train_loss=train_loss, train_acc=train_acc, test_acc=test_acc,
                       tempo_s=round(time.time() - start))

### Checagem do gradiente (aula de backpropagation)
"Na prática: sempre use o gradiente analítico, mas cheque a implementação com o gradiente numérico."
`gradcheck` compara o gradiente do autograd com diferenças finitas numa MLP pequena, em float64.

In [ ]:
torch.manual_seed(SEED)
tiny = nn.Sequential(nn.Linear(5, 4), nn.ReLU(), nn.Linear(4, 3)).double()
xg, yg = torch.randn(6, 5, dtype=torch.double), torch.randint(0, 3, (6,))
names = [n for n, _ in tiny.named_parameters()]
params = tuple(p.detach().clone().requires_grad_() for p in tiny.parameters())


def tiny_loss(*ps):
    return F.cross_entropy(functional_call(tiny, dict(zip(names, ps)), (xg,)), yg)


assert torch.autograd.gradcheck(tiny_loss, params, eps=1e-6, atol=1e-5)
print("gradiente analítico (autograd) = gradiente numérico")

## Baseline não neural: kNN (Lecture 11)

kNN nos pixels normalizados, com distância euclidiana. O K é escolhido **pela validação** e só o K
escolhido é avaliado no teste. Serve de referência para medir o ganho das redes.

In [ ]:
KS = [1, 3, 5, 10, 20]


def knn_neighbors(x_query, ref, k, chunk=1024):
    """Índices dos k vizinhos mais próximos (euclidiana) de cada consulta em `ref` (já pré-processado e
    achatado), calculados em blocos na GPU."""
    ref_sq = ref.pow(2).sum(1)
    out = []
    for i in range(0, len(x_query), chunk):
        q = preprocess(x_query[i:i + chunk], DEFAULT_PRE).flatten(1)
        dist = q.pow(2).sum(1, keepdim=True) - 2 * q @ ref.T + ref_sq  # ||q - r||², sem a raiz
        out.append(dist.topk(k, largest=False).indices)
    return torch.cat(out)


def knn_accuracy(neighbors, y_ref, y_true, k):
    votes = F.one_hot(y_ref[neighbors[:, :k]], 10).sum(1)  # empate: fica a menor classe
    return (votes.argmax(1) == y_true).float().mean().item()


knn_path = RESULTS_DIR / "knn.csv"
knn_key = dict(seed=SEED, n_treino=len(x_train))  # o cache só vale para a mesma divisão
cached = pd.read_csv(knn_path) if knn_path.exists() else None
if (cached is not None and list(cached["k"]) == KS
        and all((cached[c] == v).all() for c, v in knn_key.items() if c in cached)):
    knn_table = cached
    print(f"carregado de {knn_path}")
else:
    ref = preprocess(x_train, DEFAULT_PRE).flatten(1)  # normaliza o treino uma vez só
    nb_val = knn_neighbors(x_val, ref, max(KS))
    knn_table = pd.DataFrame({"k": KS, "val_acc": [knn_accuracy(nb_val, y_train, y_val, k) for k in KS]})
    best_k = int(knn_table.loc[knn_table["val_acc"].idxmax(), "k"])
    test_acc_knn = knn_accuracy(knn_neighbors(x_test, ref, best_k), y_train, y_test, best_k)
    del ref, nb_val
    knn_table["escolhido"] = knn_table["k"] == best_k
    knn_table["test_acc"] = knn_table["escolhido"].map({True: test_acc_knn, False: None})
    knn_table = knn_table.assign(**knn_key)
    knn_table.to_csv(knn_path, index=False)
knn_table

## Experimentos

| # | Experimento | Mudança (em relação a) | Aula |
|---|---|---|---|
| 1 | linear | classificador linear 3072 → 10, loss softmax | loss_function |
| 2 | linear_svm | loss SVM multiclasse (1) | loss_function |
| 3 | mlp_baseline | MLP 512-256, ReLU, Adam lr=1e-3, batch 64 | aula-2 |
| 4 | mlp_lr_1e-4 | lr 1e-4 (3) | backpropagation |
| 5 | mlp_lr_1e-2 | lr 1e-2 (3) | backpropagation |
| 6 | mlp_batch_32 | batch 32 (3) | backpropagation |
| 7 | mlp_batch_256 | batch 256 (3) | backpropagation |
| 8 | mlp_sgd | otimizador SGD no lugar do Adam (5, mesma lr 0,01) | backpropagation |
| 9 | mlp_sgd_momentum | + momentum 0,9 (8) | backpropagation |
| 10 | mlp_100 | uma camada oculta de 100 (3) | aula-2 |
| 11 | mlp_leaky_relu | Leaky ReLU (3) | aula-2 |
| 12 | mlp_tanh | tanh (3) | aula-2 |
| 13 | mlp_sigmoid | sigmóide (3) | aula-2 |
| 14 | mlp_sem_norm | sem normalização, só /255 (3) | aula-2 |
| 15 | mlp_so_media | só subtrai a média por canal (3) | aula-2 |
| 16 | mlp_cinza | tons de cinza, 1024 entradas (3) | aula-2 |
| 17 | mlp_l2 | regularização L2, λ = 1e-4 (3) | loss_function |
| 18 | mlp_dropout | Dropout 0,3 (3) | aula-2 |
| 19 | mlp_dropout_0.5 | Dropout 0,5 (18) | aula-2 |
| 20 | mlp_bn_dropout | + BatchNorm (18) | |
| 21 | mlp_deep | camadas 1024-512-256 (20) | |
| 22 | cnn | CNN 3 blocos 32/64/128 filtros | |
| 23 | cnn_augment | + data augmentation (22) | |
| 24 | cnn_wide | filtros 64/128/256 (23) | |
| 25 | cnn_wd | weight decay 0.05 (23) | |
| 26 | cnn_30ep | 30 épocas (23) | |
| 27 | cnn_ls | label smoothing 0.1 (23) | |
| 28 | cnn_deep | 3 convoluções por bloco em vez de 2 (23) | |
| 29 | cnn_cutout | cutout: quadrado 8×8 zerado em cada imagem do treino (23) | |
| 30 | cnn_best | (23) + as mudanças de 24–29 que melhoraram a validação | |

In [ ]:
# Configs só com dados (listas, não tuplas), para poderem ser comparadas com as salvas em JSON.
MLP = dict(arch="mlp", lr=1e-3)
CNN_AUG = dict(arch="cnn", lr=1e-3, augment=True)
CNN_VARIATIONS = {
    "cnn_wide": dict(arch_args={"widths": [64, 128, 256]}),
    "cnn_wd":   dict(weight_decay=0.05),
    "cnn_30ep": dict(epochs=2 * EPOCHS),  # 30 épocas
    "cnn_ls":   dict(label_smoothing=0.1),
    "cnn_deep":   dict(arch_args={"convs": 3}),  # #17: mais profunda em vez de mais larga
    "cnn_cutout": dict(cutout=True),             # #17: cutout 8x8 depois do augment
}
EXPERIMENTS = {
    "linear":           dict(arch="linear", lr=1e-3),
    "linear_svm":       dict(arch="linear", lr=1e-3, loss="svm"),
    "mlp_baseline":     MLP,
    "mlp_lr_1e-4":      {**MLP, "lr": 1e-4},
    "mlp_lr_1e-2":      {**MLP, "lr": 1e-2},
    "mlp_batch_32":     {**MLP, "batch": 32},
    "mlp_batch_256":    {**MLP, "batch": 256},
    "mlp_sgd":          {**MLP, "optimizer": "sgd", "lr": 1e-2},  # compara com mlp_lr_1e-2 (Adam, mesma lr)
    "mlp_sgd_momentum": {**MLP, "optimizer": "sgd", "lr": 1e-2, "momentum": 0.9},
    "mlp_100":          {**MLP, "arch_args": {"hidden": [100]}},
    "mlp_leaky_relu":   {**MLP, "arch_args": {"activation": "leaky_relu"}},
    "mlp_tanh":         {**MLP, "arch_args": {"activation": "tanh"}},
    "mlp_sigmoid":      {**MLP, "arch_args": {"activation": "sigmoid"}},
    "mlp_sem_norm":     {**MLP, "norm": "none"},
    "mlp_so_media":     {**MLP, "norm": "mean"},
    "mlp_cinza":        {**MLP, "gray": True, "arch_args": {"in_features": 32 * 32}},
    "mlp_l2":           {**MLP, "l2": 1e-4},
    "mlp_dropout":      {**MLP, "arch_args": {"dropout": 0.3}},
    "mlp_dropout_0.5":  {**MLP, "arch_args": {"dropout": 0.5}},
    "mlp_bn_dropout":   {**MLP, "arch_args": {"dropout": 0.3, "batchnorm": True}},
    "mlp_deep":         {**MLP, "arch_args": {"hidden": [1024, 512, 256], "dropout": 0.3, "batchnorm": True}},
    "cnn":              dict(arch="cnn", lr=1e-3),
    "cnn_augment":      CNN_AUG,
    **{name: {**CNN_AUG, **change} for name, change in CNN_VARIATIONS.items()},
}
results, histories, configs = {}, {}, {}


def cached_train(path, cfg, split_seed=None):
    """Treina e salva em `path`; se já foi salvo com exatamente a mesma config completa, só carrega.

    `split_seed`: treina numa divisão treino/validação sorteada com essa semente (só montada sem cache).
    """
    saved = json.loads(path.read_text()) if path.exists() else {}
    if saved.get("config") == cfg:
        print(f"{path.stem}: carregado de {path}")
        return saved
    print(f"{path.stem}:")
    _, hist, summary = train(cfg, make_split(split_seed) if split_seed is not None else None)
    saved = dict(summary=summary, hist=hist, config=cfg)
    path.write_text(json.dumps(saved))
    return saved


def run(name, cfg):
    cfg = full_config(cfg)
    saved = cached_train(RESULTS_DIR / f"{name}.json", cfg)
    results[name], histories[name], configs[name] = saved["summary"], saved["hist"], cfg
    s = saved["summary"]
    print(f"  -> val {s['val_acc']:.4f} (época {s['melhor_epoca']}) | teste {s['test_acc']:.4f}")

In [ ]:
for name, cfg in EXPERIMENTS.items():
    run(name, cfg)

### Experimento 30 — combinação
Junta à `cnn_augment` só as mudanças de 24–29 que superaram a `cnn_augment` na **validação**.

In [ ]:
improved = [n for n in CNN_VARIATIONS if results[n]["val_acc"] > results["cnn_augment"]["val_acc"]]
print("mudanças que melhoraram a validação:", improved or "nenhuma")
cnn_best = dict(CNN_AUG)
for n in improved:  # arch_args são mesclados, para uma variação não apagar a outra (ex.: widths e convs)
    change = CNN_VARIATIONS[n]
    cnn_best.update({**change, "arch_args": {**cnn_best.get("arch_args", {}), **change.get("arch_args", {})}})
if improved:
    run("cnn_best", cnn_best)
else:  # sem mudanças, cnn_best == cnn_augment: reaproveita em vez de retreinar
    for d in (results, histories, configs):
        d["cnn_best"] = d["cnn_augment"]

## Resultados

In [ ]:
table = pd.DataFrame.from_dict(results, orient="index").rename_axis("modificacao").reset_index()
table.insert(0, "experimento", range(1, len(table) + 1))
table.to_csv(RESULTS_DIR / "results.csv", index=False)
table.style.format(precision=4).highlight_max(subset=["val_acc"], color="#c6efce")

In [ ]:
def plot_curves(name):
    h = histories[name]
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, key, title in ((ax_loss, "loss", "Loss"), (ax_acc, "acc", "Acurácia")):
        epochs = range(1, len(h[f"train_{key}"]) + 1)
        ax.plot(epochs, h[f"train_{key}"], label="treino")
        ax.plot(epochs, h[f"val_{key}"], label="validação")
        ax.axvline(results[name]["melhor_epoca"], color="gray", ls=":", label="melhor época")
        ax.set(title=f"{name} — {title}", xlabel="época")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"{name}.png", dpi=120)
    plt.show()


for name in results:
    plot_curves(name)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for name in ("linear", "mlp_baseline", "mlp_bn_dropout", "cnn", "cnn_augment", "cnn_best"):
    ax.plot(range(1, len(histories[name]["val_acc"]) + 1), histories[name]["val_acc"], label=name)
ax.set(title="Acurácia de validação por época", xlabel="época", ylabel="acurácia")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "comparacao_validacao.png", dpi=120)
plt.show()

# Efeito da taxa de aprendizagem (slide da aula de backpropagation): mesma MLP com três taxas.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, lr in (("mlp_lr_1e-4", "1e-4"), ("mlp_baseline", "1e-3"), ("mlp_lr_1e-2", "1e-2")):
    for ax, key in zip(axes, ("train_loss", "val_loss")):
        ax.plot(range(1, len(histories[name][key]) + 1), histories[name][key], label=f"lr {lr}")
for ax, title in zip(axes, ("Loss de treino", "Loss de validação")):
    ax.set(title=title, xlabel="época", ylabel="loss")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "taxas_aprendizado.png", dpi=120)
plt.show()

best = table.loc[table["val_acc"].idxmax()]
print(f"Melhor na validação: {best['modificacao']} | val {best['val_acc']:.4f} | teste {best['test_acc']:.4f}")

## Random subsampling (Lecture 11)

Um único holdout pode favorecer um modelo por sorte do sorteio. Os 3 melhores na validação (com configs
distintas) são treinados de novo em outras divisões treino/validação, cada uma com sua semente, e
reportamos média ± desvio. A normalização continua com a média e o desvio da divisão principal.
A escolha do modelo final **não muda**: continua sendo a validação da divisão fixa (semente 42).

In [ ]:
N_SPLITS = 2 if SMOKE else 5
top = []
for name in table.sort_values("val_acc", ascending=False)["modificacao"]:
    if all(configs[name] != configs[t] for t in top):  # cnn_best pode ser cópia da cnn_augment
        top.append(name)
    if len(top) == 3:
        break

(RESULTS_DIR / "subsampling").mkdir(exist_ok=True)
rows = []
for name in top:
    for i in range(1, N_SPLITS + 1):
        cfg = {**configs[name], "seed": SEED + i}  # semente da divisão e da inicialização
        path = RESULTS_DIR / "subsampling" / f"{name}_divisao{i}.json"
        s = cached_train(path, cfg, split_seed=cfg["seed"])["summary"]
        rows.append(dict(modelo=name, divisao=i, val_acc=s["val_acc"], test_acc=s["test_acc"]))

subsampling = (pd.DataFrame(rows).groupby("modelo", sort=False)
               .agg(val_media=("val_acc", "mean"), val_desvio=("val_acc", "std"),
                    teste_media=("test_acc", "mean"), teste_desvio=("test_acc", "std"), divisoes=("val_acc", "size"))
               .reset_index())
subsampling.to_csv(RESULTS_DIR / "subsampling.csv", index=False)
subsampling.style.format(precision=4)

## Modelo final

O modelo final é o experimento com a **maior acurácia de validação**. Ele é retreinado com todo o
`split="train"` (treino + validação) com a mesma config e semente, parando na época em que atingiu a
melhor validação (com o mesmo agendamento de LR). Como não sobra validação, fica o modelo da última época.
O teste só informa a acurácia.

Os pesos são salvos em `WEIGHTS_PATH` junto com a config, a média e o desvio da normalização, que a
predição do conjunto adicional precisa. Se o arquivo já existe com a mesma config, não retreina.

In [ ]:
FINAL = best["modificacao"]  # o melhor na validação, escolhido na tabela de resultados
final_cfg = {**configs[FINAL], "stop_epoch": results[FINAL]["melhor_epoca"]}
print(f"Modelo final: {FINAL} | val {results[FINAL]['val_acc']:.4f} | "
      f"{final_cfg['stop_epoch']} de {final_cfg['epochs']} épocas")
print(json.dumps(final_cfg, indent=2))

try:
    ckpt = torch.load(WEIGHTS_PATH, map_location="cpu")
except Exception:  # ausente ou corrompido (ex.: Colab caiu ao salvar)
    ckpt = {}
if ckpt.get("config") == final_cfg:
    print(f"carregado de {WEIGHTS_PATH}")
else:
    trained, final_summary = train_final(final_cfg)
    ckpt = dict(experimento=FINAL, config=final_cfg, mean=MEAN.cpu(), std=STD.cpu(),
                state_dict={k: v.cpu() for k, v in trained.state_dict().items()},
                summary=final_summary, val_acc_experimento=results[FINAL]["val_acc"])
    WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = WEIGHTS_PATH.with_suffix(".tmp")
    torch.save(ckpt, tmp)
    tmp.replace(WEIGHTS_PATH)  # troca atômica: nunca fica um best.pt pela metade
    print(f"pesos salvos em {WEIGHTS_PATH}")

In [ ]:
# Confere que os pesos salvos reproduzem a acurácia de teste (é o mesmo arquivo usado na predição).
# Em outro dispositivo (GPU do Colab vs Mac/CPU) algumas predições podem mudar: tolera 0,1 ponto ou 2 imagens.
final_model = ARCHS[ckpt["config"]["arch"]](**ckpt["config"]["arch_args"]).to(DEVICE)
final_model.load_state_dict(ckpt["state_dict"])
_, test_acc = run_epoch(final_model, x_test, y_test, ckpt["config"])
tolerance = max(1e-3, 2 / len(y_test))
assert abs(test_acc - ckpt["summary"]["test_acc"]) <= tolerance, (test_acc, ckpt["summary"]["test_acc"])

final_info = dict(experimento=ckpt["experimento"], arquitetura=str(final_model), config=ckpt["config"],
                  parametros=sum(p.numel() for p in final_model.parameters()),
                  val_acc=ckpt["val_acc_experimento"], test_acc=test_acc,
                  train_acc=ckpt["summary"]["train_acc"], train_loss=ckpt["summary"]["train_loss"])
(RESULTS_DIR / "final.json").write_text(json.dumps(final_info, indent=2))
print(final_model)
print(f"{final_info['parametros']:,} parâmetros | val (experimento) {final_info['val_acc']:.4f} | teste {test_acc:.4f}")

## TTA e ensemble (#17)

Duas formas de melhorar a predição sem mudar o treino de cada modelo:
- **TTA** (test-time augmentation): média do softmax da imagem original e de 4 cópias deslocadas ±2 px na
  horizontal e na vertical. Sem espelhamento: dígito espelhado é outro símbolo.
- **Ensemble:** `N_SEEDS` modelos da config final com sementes 42 a 46 (muda a inicialização e a ordem dos
  batches; a divisão treino/validação é a mesma) e média do softmax.

O número de modelos (1 a `N_SEEDS`), com e sem TTA, é escolhido **pela validação**, com modelos treinados só no
treino; o teste só informa. O método escolhido é refeito com modelos treinados em todo o `split="train"` (o da
semente 42 é o próprio `weights/best.pt`) e é ele que gera as predições do conjunto adicional. Os pesos ficam em
`weights/ensemble/` e, como os resultados, não são retreinados se a config não mudar.


In [ ]:
SHIFTS = ((2, 0), (-2, 0), (0, 2), (0, -2))  # (dx, dy) em pixels


PAD = max(abs(d) for s in SHIFTS for d in s)


def shift(x, dx, dy):
    """Desloca o lote (já normalizado) dx px para a direita e dy para baixo, preenchendo com 0 (a cor média)."""
    h, w = x.shape[-2:]
    padded = F.pad(x, (PAD, PAD, PAD, PAD))
    return padded[..., PAD - dy: PAD - dy + h, PAD - dx: PAD - dx + w]


@torch.no_grad()
def predict_proba(models, x, cfg, tta):
    """Média do softmax dos `models` (e, com `tta`, das cópias deslocadas), em lotes de 1024."""
    for m in models:
        m.eval()
    out = []
    for xb in x.split(1024):
        xb = preprocess(xb, cfg)
        views = [xb] + ([shift(xb, dx, dy) for dx, dy in SHIFTS] if tta else [])
        out.append(torch.stack([m(v).softmax(1) for m in models for v in views]).mean(0))
    return torch.cat(out)


def train_model(cfg):
    model, _, summary = train(cfg)
    return model, summary


def cached_model(path, cfg, trainer):
    """Carrega os pesos de `path` se foram salvos com a mesma config; senão treina com `trainer(cfg)`, que
    devolve (modelo, resumo), e salva. Devolve (modelo, resumo)."""
    try:
        ck = torch.load(path, map_location="cpu")
    except Exception:  # ausente ou corrompido
        ck = {}
    if ck.get("config") == cfg:
        print(f"{path.stem}: carregado de {path}")
    else:
        print(f"{path.stem}:")
        trained, summary = trainer(cfg)
        ck = dict(config=cfg, summary=summary, state_dict={k: v.cpu() for k, v in trained.state_dict().items()})
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(".tmp")
        torch.save(ck, tmp)
        tmp.replace(path)
    model = ARCHS[cfg["arch"]](**cfg["arch_args"]).to(DEVICE)
    model.load_state_dict(ck["state_dict"])
    return model, ck["summary"]


N_SEEDS = 2 if SMOKE else 5
ENSEMBLE_DIR = WEIGHTS_PATH.parent / "ensemble"
val_runs = [cached_model(ENSEMBLE_DIR / f"val_seed{SEED + i}.pt", {**configs[FINAL], "seed": SEED + i}, train_model)
            for i in range(N_SEEDS)]
# Softmax de cada modelo calculado uma vez; o ensemble de k é a média dos k primeiros (não treina nada).
probs = {(split, tta): torch.stack([predict_proba([m], x, configs[FINAL], tta) for m, _ in val_runs])
         for split, x in (("val", x_val), ("test", x_test)) for tta in (False, True)}
rows = []
for n_models in range(1, N_SEEDS + 1):
    for tta in (False, True):
        acc = {split: (probs[split, tta][:n_models].mean(0).argmax(1) == y).float().mean().item()
               for split, y in (("val", y_val), ("test", y_test))}
        name = (f"ensemble de {n_models}" if n_models > 1 else "1 modelo") + (" + TTA" if tta else "")
        rows.append(dict(metodo=name, n_modelos=n_models, tta=tta, val_acc=acc["val"], test_acc=acc["test"]))
methods = pd.DataFrame(rows)
methods["escolhido"] = methods.index == methods["val_acc"].idxmax()  # empate: fica o mais simples (vem antes)
methods.to_csv(RESULTS_DIR / "tta_ensemble.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
for tta, label in ((False, "sem TTA"), (True, "com TTA")):
    sub = methods[methods["tta"] == tta]
    ax.plot(sub["n_modelos"], sub["val_acc"], "o-", label=label)
ax.set(title=f"Ensemble — {FINAL} (validação)", xlabel="número de modelos", ylabel="acurácia",
       xticks=range(1, N_SEEDS + 1))
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "ensemble.png", dpi=120)
plt.show()
methods.style.format(precision=4)


In [ ]:
# Refaz o método escolhido com modelos treinados em todo o split="train", como o weights/best.pt: cada semente
# para na sua melhor época de validação.
METHOD = methods[methods["escolhido"]].iloc[0]
TTA = bool(METHOD["tta"])
final_models = [final_model]  # semente 42 (weights/best.pt)
for i in range(1, int(METHOD["n_modelos"])):
    cfg_i = {**final_cfg, "seed": SEED + i, "stop_epoch": val_runs[i][1]["melhor_epoca"]}
    final_models.append(cached_model(ENSEMBLE_DIR / f"final_seed{SEED + i}.pt", cfg_i, train_final)[0])


def predict_final(x):
    return predict_proba(final_models, x, ckpt["config"], TTA).argmax(1)


preds = predict_final(x_test)  # também usadas na análise de erros
final_pred = dict(metodo=METHOD["metodo"], n_modelos=len(final_models), tta=TTA,
                  test_acc=(preds == y_test).float().mean().item())
(RESULTS_DIR / "final_predicao.json").write_text(json.dumps(final_pred, indent=2))
print(f"Predição final: {final_pred['metodo']} | teste {final_pred['test_acc']:.4f}")


## Solução final pt4 (#20): ensemble de 20 CNNs de 60 épocas + TTA

A bateria pt4 rodou nas DGX Spark com [`dgx/svhn_dgx.py`](dgx/svhn_dgx.py), com o mesmo protocolo deste notebook
(mesma divisão, normalização e escolha pela validação). Plano, resultados e decisões estão em
[`dgx/README.md`](dgx/README.md) e [`dgx/PLANO.md`](dgx/PLANO.md). O que ela escolheu pela validação:
- a CNN `cnn_best` com **60 épocas** em vez de 30;
- **20 sementes**, de 42 a 61; a curva parou de subir em 20, e 25 e 30 não ganharam;
- **TTA shift12**: a imagem e 8 cópias deslocadas 1 e 2 px na horizontal e na vertical.

Na validação, isso dá 97,52%, contra 97,31% do ensemble de 5 + TTA acima. Cada membro foi retreinado em todo o
`split="train"` até a melhor época de validação da sua semente ([`dgx/final_ensemble.py`](dgx/final_ensemble.py)).
Os pesos estão em `weights/pt4/`.

Quando os pesos existem, esta célula substitui o método da seção anterior: a análise de erros e as predições do
conjunto adicional passam a usar a solução pt4. No modo de teste (CI), a célula é pulada. Nas DGX a inferência
foi em bf16; aqui é em fp32, o que pode mudar a predição de poucas imagens.


In [ ]:
PT4_DIR = Path("weights/pt4")
SHIFTS12 = ((1, 0), (-1, 0), (0, 1), (0, -1), (2, 0), (-2, 0), (0, 2), (0, -2))  # PAD = 2 cobre os dois

if PT4_DIR.exists() and not SMOKE:
    pt4_models = []
    for f in sorted(PT4_DIR.glob("s*.pt")):
        member = torch.load(f, map_location="cpu")
        model = cnn(**member["config"]["arch_args"]).to(DEVICE)
        model.load_state_dict(member["state_dict"])
        pt4_models.append(model.eval())

    @torch.no_grad()
    def predict_final(x):
        """Média do softmax dos 20 membros na imagem e nas 8 cópias deslocadas, em lotes de 1024."""
        out = []
        for xb in x.split(1024):
            xb = preprocess(xb, DEFAULT_PRE)
            views = [xb] + [shift(xb, dx, dy) for dx, dy in SHIFTS12]
            out.append(torch.stack([m(v).softmax(1) for m in pt4_models for v in views]).mean(0))
        return torch.cat(out).argmax(1)

    preds = predict_final(x_test)
    final_pred = dict(metodo="pt4: ensemble de 20 CNNs de 60 épocas + TTA shift12", n_modelos=len(pt4_models),
                      tta=True, test_acc=(preds == y_test).float().mean().item())
    (RESULTS_DIR / "final_predicao.json").write_text(json.dumps(final_pred, indent=2))
    print(f"Predição final: {final_pred['metodo']} | {len(pt4_models)} membros | teste {final_pred['test_acc']:.4f}")


## Análise de erros do modelo final (Lecture 11)

Feita com o método que gera as predições (o modelo final, com TTA e/ou ensemble se ganharam na validação).

Matriz de confusão no teste, precisão, recall e F1 por dígito (o SVHN é desbalanceado: tem bem mais "1"
que "9", e a acurácia sozinha esconde isso) e exemplos de imagens classificadas errado.

In [ ]:
confusion = torch.bincount(y_test * 10 + preds, minlength=100).view(10, 10).cpu()  # linha = real, coluna = predito

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(confusion, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, int(confusion[i, j]), ha="center", va="center", fontsize=7,
                color="white" if confusion[i, j] > confusion.max() / 2 else "black")
ax.set(xticks=range(10), yticks=range(10), xlabel="predito", ylabel="real",
       title=f"Matriz de confusão — {final_pred['metodo']} (teste)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusao.png", dpi=120)
plt.show()

hits = confusion.diag().double()
precision = hits / confusion.sum(0).clamp(min=1)
recall = hits / confusion.sum(1).clamp(min=1)
f1 = 2 * precision * recall / (precision + recall).clamp(min=1e-12)
per_class = pd.DataFrame({"digito": range(10), "suporte": confusion.sum(1).tolist(), "precisao": precision.tolist(),
                          "recall": recall.tolist(), "f1": f1.tolist()})
per_class.to_csv(RESULTS_DIR / "metricas_por_classe.csv", index=False)

off_diag = confusion.clone().fill_diagonal_(0)
top_pairs = off_diag.flatten().topk(5)
print("Pares mais confundidos (real -> predito):")
for count, idx in zip(top_pairs.values.tolist(), top_pairs.indices.tolist()):
    print(f"  {idx // 10} -> {idx % 10}: {count} imagens")
per_class.style.format(precision=4)

In [ ]:
wrong = (preds != y_test).nonzero().flatten()[:24].tolist()
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(x_test[i].permute(1, 2, 0).cpu())
    ax.set_title(f"{int(y_test[i])} → {int(preds[i])}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("Erros do modelo final no teste (real → predito)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "erros.png", dpi=120)
plt.show()

## Predições do conjunto adicional

Só depois do modelo final definido: o conjunto do professor não entra em treino, ajuste nem escolha de modelo.
As imagens (PNG 32×32, sem rótulo) ficam em `ADICIONAL_DIR` e são lidas em ordem numérica do nome do
arquivo. O método escolhido na seção anterior (o modelo final de `weights/best.pt`, com TTA e/ou ensemble se
eles ganharem na validação), com a mesma normalização do treino, grava uma predição por linha em `RESULTS_TXT`.

In [ ]:
ADICIONAL_DIR = DATA_DIR / "adicional"  # imagens do professor: 000000.png, 000001.png, ...
RESULTS_TXT = Path("SAMUEL_WILKER_BERNARDO_RESULTS.txt")
if SMOKE:  # imagens aleatórias no mesmo formato; o 10 depois do 2 exercita a ordem numérica
    ADICIONAL_DIR, RESULTS_TXT = Path("smoke/adicional"), Path("smoke/SAMUEL_WILKER_BERNARDO_RESULTS.txt")
    ADICIONAL_DIR.mkdir(parents=True)
    g = torch.Generator().manual_seed(SEED)
    for i in (0, 1, 2, 10):
        Image.fromarray(torch.randint(0, 256, (32, 32, 3), dtype=torch.uint8, generator=g).numpy()).save(ADICIONAL_DIR / f"{i}.png")

files = sorted((f for f in ADICIONAL_DIR.glob("*") if f.suffix.lower() == ".png"), key=lambda f: int(f.stem))
if not files:  # ex.: Colab sem o upload das imagens; não é erro, só não há o que prever
    print(f"nenhuma imagem em {ADICIONAL_DIR}: predições não geradas, {RESULTS_TXT} não foi atualizado")
else:
    x_extra = torch.from_numpy(np.stack([np.asarray(Image.open(f).convert("RGB")) for f in files])).permute(0, 3, 1, 2)
    extra_preds = predict_final(x_extra.to(DEVICE)).tolist()
    RESULTS_TXT.write_text("".join(f"{d}\n" for d in extra_preds))
    print(f"{len(extra_preds)} predições de {files[0].name} a {files[-1].name} em {RESULTS_TXT}")
    display(pd.Series(extra_preds).value_counts().sort_index().rename("imagens").to_frame().T)

## Experimentos extras (#14)

Ideias fora do escopo da entrega: **não entram** na tabela principal, no random subsampling nem na escolha do
modelo final. Ficam em `RESULTS_DIR / "extras"` e em `extras.csv`, cada um comparado com a sua base.

| Extra | Mudança (em relação a) | Pergunta |
|---|---|---|
| cnn_sem_bn | sem as camadas de BatchNorm (cnn) | quanto o BN ajuda na convergência e na acurácia |
| mlp_init_xavier, mlp_init_kaiming_normal, mlp_init_normal_0.01 | inicialização dos pesos (mlp_baseline, que usa o padrão do PyTorch) | efeito na convergência e nas ativações mortas ou saturadas |
| {modelo final}\_10pct, \_25pct, \_50pct | só 10%, 25% e 50% do treino (modelo final, 100%) | mais dados ainda ajudariam ou o modelo saturou? |


In [ ]:
EXTRAS_DIR = RESULTS_DIR / "extras"
EXTRAS_DIR.mkdir(exist_ok=True)
FRACS = (10, 25, 50)  # % do treino; 100% é o próprio modelo final
EXTRAS = {  # nome: (base de comparação, config)
    "cnn_sem_bn": ("cnn", {**configs["cnn"], "arch_args": {**configs["cnn"]["arch_args"], "batchnorm": False}}),
    **{f"mlp_init_{i}": ("mlp_baseline", {**configs["mlp_baseline"],
                                          "arch_args": {**configs["mlp_baseline"]["arch_args"], "init": i}})
       for i in INITS},
    **{f"{FINAL}_{p}pct": (FINAL, {**configs[FINAL], "train_frac": p / 100}) for p in FRACS},
}
extras, extra_hist = {}, {}
for name, (base, cfg) in EXTRAS.items():
    saved = cached_train(EXTRAS_DIR / f"{name}.json", full_config(cfg))
    extras[name], extra_hist[name] = saved["summary"], saved["hist"]

extras_table = pd.DataFrame([
    dict(experimento=name, base=base, **{k: extras[name][k] for k in ("train_acc", "val_acc", "test_acc",
                                                                        "melhor_epoca", "tempo_s")},
         delta_val=extras[name]["val_acc"] - results[base]["val_acc"])
    for name, (base, _) in EXTRAS.items()])
extras_table.to_csv(RESULTS_DIR / "extras.csv", index=False)
extras_table.style.format(precision=4)


In [ ]:
# cnn_sem_bn: convergência (loss de treino) e validação por época, contra a cnn com BatchNorm.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, h in (("cnn (com BN)", histories["cnn"]), ("cnn_sem_bn", extra_hist["cnn_sem_bn"])):
    for ax, key in zip(axes, ("train_loss", "val_acc")):
        ax.plot(range(1, len(h[key]) + 1), h[key], label=name)
for ax, title in zip(axes, ("Loss de treino", "Acurácia de validação")):
    ax.set(title=title, xlabel="época")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "extras_sem_bn.png", dpi=120)
plt.show()


In [ ]:
# Inicialização: ativações das camadas ocultas na inicialização (antes de treinar), num lote de validação,
# e a validação por época. Unidade "morta" = ReLU que não ativa para nenhuma imagem do lote.
@torch.no_grad()
def activation_stats(init):
    with torch.random.fork_rng(devices=[]):  # não altera o RNG global do notebook
        torch.manual_seed(SEED)
        model = mlp(init=init).to(DEVICE).eval()
    x, stats = preprocess(x_val[:1024], DEFAULT_PRE), []
    for layer in model:
        x = layer(x)
        if isinstance(layer, nn.ReLU):
            stats.append(dict(desvio=x.std().item(), mortas=(x.amax(0) == 0).float().mean().item()))
    return stats


init_names = {"padrão": None, **{i: i for i in INITS}}
init_stats = {label: activation_stats(init) for label, init in init_names.items()}
n_layers = len(init_stats["padrão"])
print(pd.DataFrame({(label, f"camada {j + 1}"): st for label, stats in init_stats.items()
                    for j, st in enumerate(stats)}).T.to_string(float_format="{:.3f}".format))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for k, (label, init) in enumerate(init_names.items()):
    stats = init_stats[label]
    axes[0].bar([j + k * 0.2 for j in range(n_layers)], [s["desvio"] for s in stats], 0.2, label=label)
    h = histories["mlp_baseline"] if init is None else extra_hist[f"mlp_init_{init}"]
    axes[1].plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], label=label)
axes[0].set_xticks([j + 0.3 for j in range(n_layers)], [f"camada {j + 1}" for j in range(n_layers)])
for ax, title in zip(axes, ("Desvio das ativações na inicialização", "Acurácia de validação")):
    ax.set(title=title)
    ax.grid(alpha=0.3)
    ax.legend()
axes[1].set(xlabel="época")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "extras_inicializacao.png", dpi=120)
plt.show()


In [ ]:
# Curva de aprendizado do modelo final: acurácia contra o tamanho do treino.
sizes = [*FRACS, 100]
runs = [extras[f"{FINAL}_{p}pct"] for p in FRACS] + [results[FINAL]]
fig, ax = plt.subplots(figsize=(7, 4))
for key, label in (("train_acc", "treino"), ("val_acc", "validação")):
    ax.plot(sizes, [r[key] for r in runs], "o-", label=label)
ax.set(title=f"Curva de aprendizado — {FINAL}", xlabel="% do treino usado", ylabel="acurácia", xticks=sizes)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "extras_curva_aprendizado.png", dpi=120)
plt.show()
